In [ ]:
import open3d 
from shepherd.shepherd_score_utils.generate_point_cloud import (
    get_atom_coords, 
    get_atomic_vdw_radii, 
    get_molecular_surface,
    get_electrostatics,
    get_electrostatics_given_point_charges,
)
from shepherd.shepherd_score_utils.pharm_utils.pharmacophore import get_pharmacophores
from shepherd.shepherd_score_utils.conformer_generation import update_mol_coordinates

print('importing rdkit')
import rdkit
from rdkit.Chem import rdDetermineBonds

import numpy as np
import matplotlib.pyplot as plt

print('importing torch')
import torch
import torch_geometric
from torch_geometric.nn import radius_graph
import torch_scatter

import pickle
from copy import deepcopy
import os
import multiprocessing
from tqdm import tqdm


print('importing lightning')
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

from shepherd.lightning_module import LightningModule
from shepherd.datasets import HeteroDataset

import importlib

from shepherd.inference import *
from shepherd.extract import create_rdkit_molecule

In [ ]:
chkpt = '../data/shepherd_chkpts/x1x3x4_diffusion_mosesaq_20240824_submission.ckpt' # checkpoint used for evaluations in preprint
#chkpt = 'shepherd_chkpts/x1x3x4_diffusion_mosesaq_20240824_30epochs_latest.ckpt' # latest checkpoint that was trained for 2-3X longer than the original version in the preprint

chkpt = '../training/jobs/x1x3x4_diffusion_mosesaq_20240824/last.ckpt'

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

model_pl = LightningModule.load_from_checkpoint(chkpt) #
params = model_pl.params
model_pl.to(device)
model_pl.model.device = device

# Conditioning on Interaction Profiles of: Natural Products

In [ ]:
with open('../data/conformers/np/molblock_charges_NPs.pkl', 'rb') as f:
    molblocks_and_charges = pickle.load(f) # len(molblocks_and_charges) == 3

# choose which natural product
index = 0 # 0, 1, 2

mol = rdkit.Chem.MolFromMolBlock(molblocks_and_charges[index][0], removeHs = False) # target natural product
charges = np.array(molblocks_and_charges[index][1]) # xTB partial charges in implicit water
display(mol)

# extracting target interaction profiles (ESP and pharmacophores)
mol_coordinates = np.array(mol.GetConformer().GetPositions())
mol_coordinates = mol_coordinates - np.mean(mol_coordinates, axis = 0)
mol = update_mol_coordinates(mol, mol_coordinates)

# conditional targets
centers = mol.GetConformer().GetPositions()
radii = get_atomic_vdw_radii(mol)
surface = get_molecular_surface(
    centers, 
    radii, 
    params['dataset']['x3']['num_points'], 
    probe_radius = params['dataset']['probe_radius'],
    num_samples_per_atom = 20,
)

pharm_types, pharm_pos, pharm_direction = get_pharmacophores(
    mol,
    multi_vector = params['dataset']['x4']['multivectors'],
    check_access = params['dataset']['x4']['check_accessibility'],
)

electrostatics = get_electrostatics_given_point_charges(
    charges, centers, surface,
)

# Conditioning on Interaction Profiles of: PDB Ligands

In [ ]:
with open('../data/conformers/pdb/molblock_charges_pdb_lowestenergy.pkl', 'rb') as f:
    molblocks_and_charges = pickle.load(f)

# choose which PDB ligand
index = 6 # 0, 1, 2, 3, 4, 5, 6

mol = rdkit.Chem.MolFromMolBlock(molblocks_and_charges[index][0], removeHs = False) # target natural product
charges = np.array(molblocks_and_charges[index][1]) # xTB partial charges in implicit water
display(mol)

# extracting target interaction profiles (ESP and pharmacophores)
mol_coordinates = np.array(mol.GetConformer().GetPositions())
mol_coordinates = mol_coordinates - np.mean(mol_coordinates, axis = 0)
mol = update_mol_coordinates(mol, mol_coordinates)

# conditional targets
centers = mol.GetConformer().GetPositions()
radii = get_atomic_vdw_radii(mol)
surface = get_molecular_surface(
    centers, 
    radii, 
    params['dataset']['x3']['num_points'], 
    probe_radius = params['dataset']['probe_radius'],
    num_samples_per_atom = 20,
)

pharm_types, pharm_pos, pharm_direction = get_pharmacophores(
    mol,
    multi_vector = params['dataset']['x4']['multivectors'],
    check_access = params['dataset']['x4']['check_accessibility'],
)

electrostatics = get_electrostatics_given_point_charges(
    charges, centers, surface,
)

# Conditioning on Interaction Profiles of: Overlapping Fragments from Fragment Screen

In [ ]:
with open('../data/conformers/fragment_merging/fragment_merge_condition.pickle', 'rb') as f:
    fragment_merge_features = pickle.load(f)
COM = fragment_merge_features['x3']['positions'].mean(0)
fragment_merge_features['x2']['positions'] = fragment_merge_features['x2']['positions'] - COM
fragment_merge_features['x3']['positions'] = fragment_merge_features['x3']['positions'] - COM
fragment_merge_features['x4']['positions'] = fragment_merge_features['x4']['positions'] - COM

# conditional targets
surface = deepcopy(fragment_merge_features['x3']['positions'])
electrostatics = deepcopy(fragment_merge_features['x3']['charges'])
pharm_types = deepcopy(fragment_merge_features['x4']['types'])
pharm_pos = deepcopy(fragment_merge_features['x4']['positions'])
pharm_direction = deepcopy(fragment_merge_features['x4']['directions'])

# Running conditional generation via inpainting

In [ ]:
n_atoms = 70
batch_size = 5
num_pharmacophores = len(pharm_types) # must equal pharm_pos.shape[0] if inpainting

In [ ]:
# 使用inference_sample函数生成样本
generated_samples = inference_sample(
    model_pl,  # 模型对象
    batch_size = batch_size,  # 批处理大小
    
    N_x1 = n_atoms,  # x1表示原子数量
    N_x4 = num_pharmacophores,  # x4表示药效团数量
    
    unconditional = False,  # 是否为无条件生成,False表示有条件生成
    
    prior_noise_scale = 1.0,  # 先验噪声的缩放因子
    denoising_noise_scale = 1.0,  # 去噪过程中噪声的缩放因子
    
    # 在特定时间步注入噪声的参数
    inject_noise_at_ts = [],  # 注入噪声的时间步列表
    inject_noise_scales = [],  # 对应的噪声缩放因子列表
    
    # 是否进行谐波化处理的参数
    harmonize = False,  # 是否启用谐波化
    harmonize_ts = [],  # 进行谐波化的时间步
    harmonize_jumps = [],  # 谐波化的跳跃步长
    
    
    # 以下参数仅在unconditional=False时有效
    
    # x2位置的修复参数(x2通过x3隐式建模)
    inpaint_x2_pos = False,  
    
    # x3(原子团)的修复参数
    inpaint_x3_pos = True,  # 是否修复x3的位置
    inpaint_x3_x = True,  # 是否修复x3的特征
    
    # x4(药效团)的修复参数
    inpaint_x4_pos = True,  # 是否修复x4的位置
    inpaint_x4_direction = True,  # 是否修复x4的方向
    inpaint_x4_type = True,  # 是否修复x4的类型
    
    # 各组件停止修复的时间点和噪声添加参数
    stop_inpainting_at_time_x2 = 0.0,  # x2停止修复的时间点
    add_noise_to_inpainted_x2_pos = 0.0,  # 对修复后的x2位置添加的噪声
    
    stop_inpainting_at_time_x3 = 0.0,  # x3停止修复的时间点
    add_noise_to_inpainted_x3_pos = 0.0,  # 对修复后的x3位置添加的噪声
    add_noise_to_inpainted_x3_x = 0.0,  # 对修复后的x3特征添加的噪声
    
    stop_inpainting_at_time_x4 = 0.0,  # x4停止修复的时间点
    add_noise_to_inpainted_x4_pos = 0.0,  # 对修复后的x4位置添加的噪声
    add_noise_to_inpainted_x4_direction = 0.0,  # 对修复后的x4方向添加的噪声
    add_noise_to_inpainted_x4_type = 0.0,  # 对修复后的x4类型添加的噪声
    
    # 修复目标值
    center_of_mass = np.zeros(3),  # x1的质心(已经中心化为零)
    surface = surface,  # 表面特征
    electrostatics = electrostatics,  # 静电特征
    pharm_types = pharm_types,  # 药效团类型
    pharm_pos = pharm_pos,  # 药效团位置
    pharm_direction = pharm_direction,  # 药效团方向
)

In [ ]:
len(generated_samples) # == batch_size

In [ ]:
generated_samples[0]['x1']['atoms']

In [ ]:
generated_samples[0]['x1']['positions']

In [ ]:
# quick visualization of generated samples
# full analyses, including extensive validity checks, can be performed by following https://github.com/coleygroup/shepherd-score

for b, sample_dict in enumerate(generated_samples):
    
    mol_ = create_rdkit_molecule(sample_dict)

    if mol_ is None:
        continue

    display(rdkit.Chem.MolFromSmiles(rdkit.Chem.MolToSmiles(mol_)))